# 04 Memory and Spill

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand how Spark uses memory on executors and the driver, what <b>spill</b> is and why it slows jobs down, the usual causes of out-of-memory (OOM) errors, and the fixes: smaller partitions, less data per row, avoiding memory-hungry operations, and keeping large results off the driver.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"Executor lost", "Container killed for exceeding memory limits" and "Driver is temporarily unavailable" are among the most common production failures, and spill is the most common hidden slowdown. Almost all of them come from a handful of patterns you can recognize and fix in code, without simply buying bigger machines.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Executor memory (classic compute)</b><br>
<code>`</code><br>Executor JVM heap (spark.executor.memory)<br>├── Reserved memory ............ ~300 MB (Spark internals)<br>├── Unified memory  ............ spark.memory.fraction (0.6) of the rest<br>│     ├── Execution: shuffles, joins, sorts, aggregations   ┐ can borrow<br>│     └── Storage:   cached data, broadcast variables        ┘ from each other<br>└── User memory ................ your objects, UDF data structures, metadata<br>+ Overhead (off-heap): spark.executor.memoryOverhead, Python workers, Arrow buffers, native (Photon)<br><code>`</code>
<ul><li><b>Spill</b>: when execution memory is full during a sort, aggregation or join, Spark writes sorted runs to local disk and merges them later. The job continues, but slowly</li><li><b>OOM</b>: when even spilling can't help (one huge row group, one enormous key, a giant collect), the task, the executor or the driver fails</li><li>On serverless, Databricks manages these settings. You fix memory problems through <b>code and data layout</b></li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A nightly aggregation used <code>collect_list</code> per customer to build purchase histories. One B2B customer had 40 million orders, so one task tried to hold all of them in a single list: spill, then "Container killed by YARN for exceeding memory limits", and three retries before failing. Replacing the full list with bounded aggregates (last 100 orders, counts and sums) made the job finish in minutes with no memory errors.
</div>

## Causes and fixes at a glance

| Symptom | Common cause | Fix |
|---|---|---|
| Spill in sorts, joins, aggregations | Partitions too big (too few shuffle partitions) | More shuffle partitions, AQE (on), filter and select early |
| One task spills or OOMs | Skewed key (lesson 02) | AQE skew join, salting, broadcast |
| Executor OOM in aggregation | `collect_list` / `collect_set` on huge groups | Bounded aggregates, two-phase, avoid collecting |
| Executor OOM after `explode` | Row explosion (arrays with millions of elements) | Filter before exploding, higher-order functions |
| Broadcast OOM / timeout | "Small" side isn't small | Remove the broadcast hint, filter the side first |
| Python worker OOM | Large pandas UDF batches, big objects in UDFs | Smaller Arrow batches, process per partition |
| **Driver** OOM | `collect()`, `toPandas()`, huge `display()`, huge broadcast | Aggregate or limit first, write results to tables |

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates test data, and a helper that times a query and reports how many partitions held data after the last shuffle.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output.
</div>

In [0]:
import time
from pyspark.sql import functions as F

events = spark.range(0, 3_000_000, 1, numPartitions=8).select(
    (F.col("id") % 50_000).alias("customer_id"),
    F.sha2(F.col("id").cast("string"), 256).alias("payload"),
    (F.rand(seed=1) * 100).alias("amount"),
)

def timed(label, df):
    start = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:<50} {time.time() - start:6.2f}s")

def try_set(key, value):
    try:
        spark.conf.set(key, value)
        return True
    except Exception:
        return False

## 1. Partition size drives memory pressure

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs a heavy aggregation (collecting payloads per customer) with very few shuffle partitions and with many, with AQE coalescing disabled where allowed.<br><br>
<b>Why it matters</b><br>With 2 partitions, each task must hold half of all customers' payloads in memory, which is where spill and OOM start. With more partitions, each task holds less. On classic compute, check the <b>Spill</b> columns of the stage in the Spark UI; on serverless, the <b>spill</b> metric in the query profile.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
On serverless, shuffle partition and AQE settings may be managed. If they can't be changed, both runs use the platform's choice and the timings will be similar.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two timings. At this demo size they can be close, because everything still fits in memory. On production-size data, the 2-partition run is the one that spills (look for the spill metric in the profile) and slows down sharply.
</div>

In [0]:
original = spark.conf.get("spark.sql.shuffle.partitions")
try_set("spark.sql.adaptive.coalescePartitions.enabled", "false")

heavy = events.groupBy("customer_id").agg(F.collect_list("payload").alias("payloads"), F.sum("amount").alias("total"))
for n in ["2", "64"]:
    if try_set("spark.sql.shuffle.partitions", n):
        timed(f"collect_list per customer, shuffle partitions = {n}", heavy)

try_set("spark.sql.shuffle.partitions", original)
try_set("spark.sql.adaptive.coalescePartitions.enabled", "true")

## 2. Don't collect what you don't need

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Compares building full lists per customer with bounded alternatives that answer the same business question.<br><br>
<b>Why it matters</b><br><code>collect_list</code> keeps every value of a group in memory in one task. Usually the business needs a summary (count, sum, latest N, distinct count), which Spark can compute with small, fixed-size state per group.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The pure fixed-size aggregates are the fastest, and their output is much smaller (fixed-size columns instead of long arrays). At this demo size the gap is modest. With millions of values per group, only the fixed-size version stays safe from OOM.
</div>

In [0]:
full = events.groupBy("customer_id").agg(F.collect_list("amount").alias("all_amounts"))
bounded = events.groupBy("customer_id").agg(
    F.count("*").alias("n"),
    F.round(F.sum("amount"), 2).alias("total"),
    F.slice(F.sort_array(F.collect_list("amount"), asc=False), 1, 5).alias("top5"),   # still collects, see below
    F.max("amount").alias("max_amount"),
)
best = events.groupBy("customer_id").agg(F.count("*").alias("n"), F.round(F.sum("amount"), 2).alias("total"),
                                         F.max("amount").alias("max_amount"))
timed("full list per customer", full)
timed("list sorted then sliced (still collects all)", bounded)
timed("pure fixed-size aggregates", best)

## 3. Row explosion

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explodes an array column and compares the row count before and after, then computes the same result without exploding.<br><br>
<b>Why it matters</b><br><code>explode</code> multiplies rows: 1 million rows with 100-element arrays become 100 million rows, which then flow through every following shuffle and join. Higher-order functions (<code>aggregate</code>, <code>filter</code>, <code>exists</code>) compute per-row results without exploding.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
50 times more rows after <code>explode</code>, and the same total from both versions. Here the explode version may even be faster, because nothing is shuffled afterwards. The cost of <code>explode</code> shows when the multiplied rows go through a later shuffle, join or write: avoid creating them when a per-row result is all you need.
</div>

In [0]:
with_arrays = spark.range(200_000).select("id", F.sequence(F.lit(1), F.lit(50)).alias("items"))
exploded = with_arrays.select("id", F.explode("items").alias("item"))
print("rows before:", with_arrays.count(), "| rows after explode:", exploded.count())

via_explode = exploded.agg(F.sum("item")).first()[0]
via_hof = with_arrays.agg(F.sum(F.aggregate("items", F.lit(0).cast("long"), lambda acc, x: acc + x))).first()[0]
print("same total:", via_explode == via_hof)
timed("sum via explode", exploded.agg(F.sum("item")))
timed("sum via aggregate() (no explode)", with_arrays.agg(F.sum(F.aggregate("items", F.lit(0).cast("long"), lambda acc, x: acc + x))))

## 4. Protect the driver

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the safe and unsafe ways to get results out of Spark.<br><br>
<b>Why it matters</b><br>The driver (or, on serverless, the client session) has limited memory. Results that come back to it must be <b>small</b>. Large results belong in tables or files.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A small aggregate and a sample are returned. The "unsafe" line is commented out on purpose.
</div>

In [0]:
summary = events.groupBy((F.col("customer_id") % 10).alias("bucket")).agg(F.sum("amount").alias("total")).collect()   # 10 rows: safe
print("aggregated result rows:", len(summary))
print("sample rows:", len(events.limit(5).collect()))                                                                    # bounded: safe
# events.toPandas()        # UNSAFE: 3 million rows with 64-character payloads into driver memory
# events.collect()         # UNSAFE for the same reason
# Instead: events.write.saveAsTable("...")   or   aggregate first, then toPandas() the small result

## 5. Python worker memory (pandas UDFs)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows how to control the size of the Arrow batches that pandas UDFs receive, and runs a UDF with a smaller batch size.<br><br>
<b>Why it matters</b><br>A pandas UDF receives batches of rows (10,000 by default). Wide rows or memory-heavy Python code can exhaust the Python worker's memory. Smaller batches reduce the peak memory per call.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The UDF runs with a batch size of 2,000 (if the setting is allowed here), and the maximum batch length seen inside the UDF is reported.
</div>

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

ok = try_set("spark.sql.execution.arrow.maxRecordsPerBatch", "2000")

@pandas_udf("long")
def batch_len(s: pd.Series) -> pd.Series:
    return pd.Series([len(s)] * len(s))

max_batch = spark.range(0, 20_000, 1, numPartitions=2).select(batch_len("id").alias("n")).agg(F.max("n")).first()[0]
print("batch size setting applied:", ok, "| largest batch seen in the UDF:", max_batch)
try_set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")

## Under the hood

```
sort / aggregate / join in a task
   ├─ fits in execution memory ─▶ done in memory (fast)
   └─ doesn't fit ─▶ spill: write sorted runs to local disk, merge later (slow, disk I/O)
                      └─ a single key/row group still too big ─▶ OOM ─▶ task retried ─▶ stage fails

driver: receives results of collect/toPandas/show, builds broadcast tables, holds plans and metadata
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> spill happens in operators after shuffles: sorts, hash aggregates, sort-merge and shuffled hash joins.

**Tuning order:** first reduce the data (filter, select, aggregate earlier), then fix skew, then adjust partitioning (shuffle partitions, AQE), and only then add memory or bigger nodes (classic compute).

In [0]:
heavy.explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: "Container killed ... exceeding memory limits"</b><br>
Off-heap or Python memory exceeded (classic compute). Reduce batch sizes and per-row objects, increase <code>spark.executor.memoryOverhead</code>, or fix skew.<br><br>
**⛔ Problem: <code>java.lang.OutOfMemoryError: Java heap space</code> in a task**<br>Partition or key too big. More partitions, fix skew, avoid <code>collect_list</code> on huge groups.<br><br>
<b>⛔ Problem: driver OOM / "Driver is temporarily unavailable"</b><br><code>collect()</code> or <code>toPandas()</code> on large data, or a huge broadcast. Aggregate first, or write to a table.<br><br>
<b>⛔ Problem: jobs are slow with lots of spill but no failure</b><br>Partitions too large. Increase shuffle partitions, or let AQE pick sizes, and shuffle less data.<br><br>
<b>⛔ Problem: caching made things worse (classic compute)</b><br>Cached data takes memory from execution. Cache only what's reused, and unpersist.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is spill in Spark?</b><br>
When a task's execution memory is full during a sort, aggregation or join, Spark writes intermediate data to local disk and merges it later. The job continues but slows down because of disk I/O and serialization.<br><br>

<b>🎤 2. How is executor memory organized?</b><br>
Part of the heap is reserved, a unified region (60% of the rest by default) is shared by execution (shuffles, joins, sorts, aggregations) and storage (cache, broadcasts), which can borrow from each other, and the remainder is user memory. Off-heap overhead covers Python workers, Arrow buffers and native memory.<br><br>

<b>🎤 3. What causes executor OOM errors, and how do you fix them?</b><br>
Oversized partitions, skewed keys, collecting huge groups (<code>collect_list</code>), exploding arrays, large broadcasts and memory-heavy UDFs. I reduce data per task (filter early, more partitions), fix skew, replace unbounded collections with bounded aggregates, avoid explosions, and adjust UDF batch sizes, before adding memory.<br><br>

<b>🎤 4. What causes driver OOM?</b><br>
Bringing too much data to the driver with <code>collect()</code>, <code>toPandas()</code> or large <code>display()</code> calls, broadcasting big tables, or very large plans and file listings. Keep driver results small and write large outputs to tables.<br><br>

<b>🎤 5. How do you see spill?</b><br>
In the Spark UI's stage metrics ("Spill (memory)" and "Spill (disk)") on classic compute, and in the query profile's per-operator spill metrics on serverless and SQL warehouses.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A notebook fails with an out-of-memory error on the driver right after <code>df.toPandas()</code> on a 200 GB DataFrame. What is the best fix?</b><br>
A. Increase the number of executors<br>
B. Aggregate or filter in Spark first, or write the result to a table instead of collecting it<br>
C. Increase <code>spark.sql.shuffle.partitions</code><br>
D. Enable change data feed<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A query profile shows a large amount of spill in a sort operator after a shuffle. What is a reasonable first change?</b><br>
A. Reduce the number of shuffle partitions to 1<br>
B. Increase the number of shuffle partitions (or let AQE size them) and reduce the data shuffled<br>
C. Disable Photon<br>
D. Convert the table to CSV<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write an aggregation that builds the full list of payloads per <code>customer_id % 10</code> (only 10 huge groups). Explain why it's risky</li><li>Rewrite it to return, per group, the count, the number of distinct payloads (approximate) and the 3 largest amounts, without keeping every payload</li><li>Explode an array of 100 elements per row for 100,000 rows, and compute the average element per row without exploding</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Ten huge groups: each task holds ~300,000 payloads in one list -> spill / OOM risk at scale
risky = events.groupBy((F.col("customer_id") % 10).alias("g")).agg(F.collect_list("payload").alias("payloads"))

# 2. Bounded state per group: counts, approximate distinct count, top 3 by max_by-style ranking
from pyspark.sql import Window
top3 = (events.withColumn("g", F.col("customer_id") % 10)
        .withColumn("rn", F.row_number().over(Window.partitionBy("g").orderBy(F.desc("amount"))))
        .filter("rn <= 3").groupBy("g").agg(F.sort_array(F.collect_list("amount"), asc=False).alias("top3")))
safe = (events.withColumn("g", F.col("customer_id") % 10)
        .groupBy("g").agg(F.count("*").alias("n"), F.approx_count_distinct("payload").alias("approx_distinct")   # within a few % of the true 300,000)
        .join(top3, "g"))
safe.orderBy("g").show(truncate=False)

# 3. Average element per row without explode
arr = spark.range(100_000).select("id", F.sequence(F.lit(1), F.lit(100)).alias("xs"))
arr.select("id", (F.aggregate("xs", F.lit(0).cast("long"), lambda a, x: a + x) / F.size("xs")).alias("avg_x")).show(3)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Executors: unified memory for execution and storage, plus user memory and off-heap overhead</li><li>Spill = execution memory full, so data goes to disk: slow but survivable. OOM = failure</li><li>Main causes: big partitions, skew, unbounded <code>collect_*</code>, row explosion, big broadcasts, heavy UDFs, collecting to the driver</li><li>Fix in this order: less data, no skew, better partitioning, then more memory (classic)</li><li>Keep driver results small. Write big results to tables</li></ul>
</div>

| Problem | First fix |
|---|---|
| Spill | More shuffle partitions, shuffle less data |
| Skewed task OOM | AQE skew join, salting, broadcast |
| `collect_list` OOM | Bounded aggregates |
| Explode blow-up | Higher-order functions |
| Driver OOM | Aggregate first, `limit`, write to tables |
| Python OOM | `spark.sql.execution.arrow.maxRecordsPerBatch` |

## Git commit

```
git add 05_optimization/04_memory_and_spill.ipynb
git commit -m "add 05_04 memory and spill notebook"
```